# SpiNNaker Synfire Example
This is an example of a Synfire Chain build on SpiNNaker, using the PyNN Neural Network language.

First we import the required libraries.

In [ ]:
from spinnaker2 import snn, helpers, hardware

Here we set up some constants that will be useful in our script

In [ ]:
n_neurons = 100
n_populations = 10
weights = 1
delays = 7
simtime = 1000

Next we must indicate that we want to run on hardware, and then set up a network that we want to start building.

In [ ]:
hw = hardware.SpiNNcloud48NodeBoard()
net = snn.Network("Synfire Example")

A group of neurons is called a Population.  Each Population has a number of neurons, and a model of the neurons being simulated.  The parameters are used to control its behaviour; most neuron models accept an ```i_offset``` parameter, which defines a constant input current to the neuron, in nano-Amps.

There are several models, e.g. ```lif_curr_exp```, ```alif```, and ```izhikavich```, as well as some 'input' models, which provide stimulation, e.g. ```spike_list``` and ```poisson_process```.

Here we create a stimulation Population from a ```spike_list``` model with a single neuron.  This will send a spike at time 0, i.e. the start of the simulation.

In [ ]:
stimulus = snn.Population(size=1, neuron_model="spike_list", params={0: [0]}, name='stimulus')
net.add(stimulus)

We next create a list of Population objects, which will become the synfire chain.  Each of these is of type ```lif_curr_exp```, which means a Leaky-Integrate-and-Fire neuron model, with one excitatory and one inhibitory exponentially-decaying, current-based synaptic input per neuron.

In [ ]:
params = {
    "threshold": 10.0,  # fire when v crosses this value
    "alpha_decay": 0.9,  # membrane leak: v *= 0.9 every time step
    "i_offset": 0.0,  # constant background current
    "v_init": 0.0,  # initial membrane voltage
    "v_reset": 0.0,  # voltage to reset to (only for reset_to_v_reset)
    "reset": "reset_to_v_reset",  # or "reset_by_subtraction"
}
chain_pops = [
    snn.Population(size=n_neurons, neuron_model="lif_curr_exp", params=params, name='chain_{}'.format(i))
    for i in range(n_populations)
]
net.add(*chain_pops)

As we would like to see the output spikes from each population, we can ask for them to be recorded.  It is also possible to record the membrane voltage using the parameter ```"v"```.

In [ ]:
for pop in chain_pops:
    pop.record = ["spikes"]

We now need to connect our Population objects together so that the spikes from one Population cause the next to spike and so on, in a loop.  Additionally, the last Population is connected to cause the first to spike again.

Populations are connected together using a Projection object.  This takes the source and target Population objects as arguments; note that the Projections are uni-directional, though a Projection can be created for each direction if a bi-directional relationship is required.

The Projection additionally requires a list of connections, which describes the connections between the individual neurons in the source Population to those in the target Population. The weight and delay of each synapse is also specified in this list.

In the example here, select a random 10 pre-neurons for each post-neuron.

In [ ]:
from random import sample
connections = [[i, j, weights, delays] for j in range(n_neurons) for i in sample(range(0, n_neurons), 10)]
for i in range(n_populations):
    proj = snn.Projection(
        pre=chain_pops[i], post=chain_pops[(i + 1) % n_populations], 
        connections=connections)
    net.add(proj)

We additionally create a Projection from the stimulus Population to the first of the chain Populations.  Without this, there is no other source of spikes in the network, so nothing would happen.

In [ ]:
stim_proj = snn.Projection(pre=stimulus, post=chain_pops[0], connections=[[0, i, 15, 0] for i in range(n_neurons)])
net.add(stim_proj)

The neural network has now been defined.  Up to this point, no code has been run on SpiNNaker.  The next step is to run the network, which will perform all the steps needed to convert the PyNN network into a running SpiNNaker network.

The run call takes a single parameter, which is the duration of the simulation in milliseconds.

In [ ]:
hw.run(net, simtime, debug=True)

Once the simulation has run, the recorded data can be retrieved and processed.  Reading back the spikes is simply done by calling ```get_spikes``` on the Population object.  Recorded spikes are returned as a dictionary of neuron id to list of spikes for that neuron if there are any, otherwise the neuron id will not be present.

In the example below, the spikes from all the Population objects are processed into a list, and an offset is calculated so that they can be drawn on a single graph later.

In [ ]:
spike_trains = [pop.get_spikes() for pop in chain_pops]
offsets = list(range(0, n_neurons * n_populations, n_neurons))

Once the data has been extracted, it is often useful to plot it.  The matplotlib library is commonly used for this.  The matplotlib ```show()``` function must be called at the end to display the results, though they can also be saved using ```savefig```, passing in a filename to save the output to.  

In [ ]:
import matplotlib.pyplot as plt
import numpy

colours = ['b', 'g', 'r', 'c', 'm', 'y', 'k']

plt.figure()
plt.xlim(0, simtime)
plt.ylim(0, n_neurons * n_populations)
offset = 0
for i, pop_spikes in enumerate(spike_trains):
    colour = colours[i % len(colours)]
    for pop_neuron_id, spikes in pop_spikes.items():
        neuron_id = pop_neuron_id + offsets[i]
        plt.plot(spikes, numpy.ones_like(spikes) * neuron_id, ".", color=colour)
plt.show()